In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/sample_data/final.csv")

In [ ]:
df = df.drop(columns = 'Unnamed: 0')

In [ ]:
df.shape

In [ ]:
df['question_text'].duplicated().sum()

In [ ]:
df1 = df[~df['question_text'].duplicated()]

In [ ]:
df1['question_text'].duplicated().sum()

In [ ]:
df1["success_rate"] = df1["num_students_correct"] / df1["num_students_attempted"]
df1["log_attempts"] = np.log1p(df1["num_students_attempted"])
df1["question_length"] = df1["question_text"].apply(lambda x: len(str(x).split()))

In [ ]:
df1.sample(100)

In [ ]:
import seaborn as sns
sns.heatmap(df1.select_dtypes('number').corr(), cmap = 'RdYlGn' ,annot = True)

In [ ]:
df1['num_students_correct'].isnull().sum()

In [ ]:
df1.isna().sum()

In [ ]:

num_cols = [
    "avg_score",
    "correct_percentage",
    "num_students_attempted",
    "num_students_correct",
    "time_taken_minutes",
    "success_rate",
    "log_attempts",
    "question_length"
]

for col in num_cols:
    df1[col].fillna(df1[col].median(), inplace=True)

In [ ]:
df1.isna().sum()

In [ ]:
X = df1.drop(columns=["bloom_level", "difficulty"])

In [ ]:
y_bloom = df1["bloom_level"]
y_difficulty = df1["difficulty"]

In [ ]:
text_col = "question_text"

cat_cols = [
    "subject",
    "topic"
]

num_cols = [
    "avg_score",
    "correct_percentage",
    "num_students_attempted",
    "num_students_correct",
    "time_taken_minutes",
    "success_rate",
    "log_attempts",
    "question_length"
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer

In [ ]:
print(X.shape)
print(y_bloom.shape)
print(y_difficulty.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, yb_train, yb_test = train_test_split(
    X, y_bloom, test_size=0.2, random_state=42
)

_, _, yd_train, yd_test = train_test_split(
    X, y_difficulty, test_size=0.2, random_state=42
)

In [ ]:
# tfidf = TfidfVectorizer(
#     stop_words="english",
#     ngram_range=(1, 2),
#     max_features=8000
# )

# X_train_text = tfidf.fit_transform(X_train["question_text"])
# X_test_text = tfidf.transform(X_test["question_text"])

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings...")

X_train_text = model.encode(
    X_train["question_text"].astype(str).tolist(),
    show_progress_bar=True
)

X_test_text = model.encode(
    X_test["question_text"].astype(str).tolist(),
    show_progress_bar=True
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore")

X_train_cat = ohe.fit_transform(X_train[["subject", "topic"]])
X_test_cat = ohe.transform(X_test[["subject", "topic"]])

In [ ]:
X_train_num = X_train[num_cols].values
X_test_num = X_test[num_cols].values

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

In [ ]:
from scipy.sparse import hstack

X_train_final = hstack([X_train_text, X_train_cat, X_train_num_scaled])
X_test_final = hstack([X_test_text, X_test_cat, X_test_num_scaled])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

bloom_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)
bloom_model.fit(X_train_final, yb_train)

pred = bloom_model.predict(X_test_final)
print(classification_report(yb_test, pred))

              precision    recall  f1-score   support

     Analyze       0.33      0.33      0.33       165
       Apply       0.30      0.27      0.28       189
      Create       0.29      0.34      0.31       158
    Evaluate       0.34      0.39      0.37       160
    Remember       0.44      0.38      0.41       200
  Understand       0.28      0.27      0.27       168

    accuracy                           0.33      1040
   macro avg       0.33      0.33      0.33      1040
weighted avg       0.33      0.33      0.33      1040



In [ ]:
difficulty_model = LogisticRegression(max_iter=3000)
difficulty_model.fit(X_train_final, yd_train)

pred = difficulty_model.predict(X_test_final)
print(classification_report(yd_test, pred))

              precision    recall  f1-score   support

        Easy       0.47      0.41      0.43       313
        Hard       0.47      0.40      0.43       263
      Medium       0.41      0.43      0.42       246
    Moderate       0.27      0.35      0.30       218

    accuracy                           0.40      1040
   macro avg       0.40      0.40      0.40      1040
weighted avg       0.41      0.40      0.40      1040



In [ ]:
yb_train

,bloom_level
4540,Evaluate
5036,Apply
3373,Evaluate
4138,Understand
283,Understand
...,...
466,Evaluate
3092,Apply
3773,Evaluate
5192,Understand


In [ ]:
yb_train.unique()

array(['Evaluate', 'Apply', 'Understand', 'Analyze', 'Create', 'Remember'],
      dtype=object)

In [ ]:
yb_train_ec = yb_train.map({'Evaluate':0, 'Apply':1, 'Understand':2, 'Analyze':3, 'Create':4, 'Remember':5})
yb_test_ec = yb_test.map({'Evaluate':0, 'Apply':1, 'Understand':2, 'Analyze':3, 'Create':4, 'Remember':5})

In [ ]:
from xgboost import XGBClassifier
clf = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

clf.fit(X_train_final,yb_train_ec)

In [ ]:
print(classification_report(yb_test_ec, clf.predict(X_test_final)))

              precision    recall  f1-score   support

           0       0.29      0.36      0.32       160
           1       0.31      0.32      0.32       189
           2       0.30      0.23      0.26       168
           3       0.33      0.35      0.34       165
           4       0.24      0.26      0.25       158
           5       0.43      0.36      0.39       200

    accuracy                           0.32      1040
   macro avg       0.32      0.31      0.31      1040
weighted avg       0.32      0.32      0.32      1040



In [ ]:
yd_train.unique()

array(['Hard', 'Moderate', 'Easy', 'Medium'], dtype=object)

In [ ]:
yd_train = yd_train.str.strip().str.capitalize()
yd_test = yd_test.str.strip().str.capitalize()

In [ ]:
label_map = {'Hard':0, 'Moderate':1, 'Easy':2, 'Medium':3}

yd_train_ec = yd_train.map(label_map)
yd_test_ec = yd_test.map(label_map)

In [ ]:
print(yd_train_ec.isna().sum())
print(yd_test_ec.isna().sum())

In [ ]:
clf = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

clf.fit(X_train_final,yd_train_ec)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
print(classification_report(yd_test_ec, clf.predict(X_test_final)))

              precision    recall  f1-score   support

           0       0.41      0.46      0.44       263
           1       0.28      0.23      0.25       218
           2       0.50      0.47      0.48       313
           3       0.40      0.44      0.42       246

    accuracy                           0.41      1040
   macro avg       0.40      0.40      0.40      1040
weighted avg       0.41      0.41      0.41      1040

